# 12-Qubit Google Supremacy Circuit: Tetron MBQC vs Direct Gate

This adjusted notebook validates the tetron MBQC translation by comparing the **final logical state** against the direct 12-qubit gate circuit.

What this notebook does:

1. **Load** one of Google's 12-qubit random-circuit-sampling files
   (`circuit_n12_m14_s0_e0_pEFGH.py`).
2. **Translate** every Cirq operation into a 24-site tetron MBQC circuit on an
   8x3 layout, using `qubit_mapping.py` and `mbqc_translated_gates.py`.
3. **Build a direct 12-qubit Qiskit reference** that applies the same
   `sqrt(X)`, `sqrt(Y)`, `sqrt(W)`, `fSim(pi/2, pi/6)` gates as ideal unitaries.
4. **Remove only the final output measurements** and compute the direct-circuit
   final statevector.
5. **Run the MBQC circuit with a statevector simulator**, keeping its mid-circuit
   measurements and feed-forward. Each run gives one post-measurement trajectory.
6. **Compare only the logical/data subsystem** of the 24-qubit MBQC state to the
   12-qubit direct state using

   $$F=\langle \psi_{\rm direct}|\rho_{\rm MBQC,data}|\psi_{\rm direct}\rangle.$$

Conventions used:

- `QUBIT_ORDER` in the Google file is `[(3,3), (3,4), ..., (5,6)]`. Direct-circuit
  qubit `i` corresponds to `QUBIT_ORDER[i]`.
- Every two-qubit gate becomes `fSim(theta=pi/2, phi=pi/6)`. The actual Cirq
  angles in the file vary slightly around those values, but the tetron MBQC
  two-qubit gadget is calibrated at this fixed pair.
- The per-edge `Rz(...)` wrappers in the Cirq file are **dropped**, in both
  circuits, so the unitary comparison is internally consistent.
- No full density matrix is built. The reduced-subsystem fidelity is evaluated
  directly from the full 24-qubit post-measurement statevector.


## 1. Imports and configuration

In [ ]:
import os, sys, importlib.util

import numpy as np
import matplotlib.pyplot as plt
import cirq

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.compiler import transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

# This notebook lives at the repo root. Make the tetron helpers importable.
REPO_ROOT  = os.path.abspath('.')
TETRON_DIR = os.path.join(REPO_ROOT, 'src', 'tetron')
GOOGLE_DIR = os.path.join(REPO_ROOT, 'google_supremacy_circuit_files')

if TETRON_DIR not in sys.path:
    sys.path.insert(0, TETRON_DIR)

from qubit_mapping import (
    grid_to_qiskit_index,
    grid_to_sq_ancilla_index,
    grid_edge_to_qiskit_indices,
    GRID_TO_LOGICAL,
)
from mbqc_translated_gates import MBQCTranslatedGates, fSim_matrix, sqrt_W_matrix

print('Imports OK.  Tetron dir:', TETRON_DIR)


## 2. Load a Google supremacy circuit

In [ ]:
CIRCUIT_FILE = os.path.join(
    GOOGLE_DIR, 'circuit_n12_m14_s0_e0_pEFGH.py'
)

def load_cirq_circuit(path):
    """exec() a Google circuit data file and return (QUBIT_ORDER, CIRCUIT)."""
    spec = importlib.util.spec_from_file_location('google_circuit', path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.QUBIT_ORDER, mod.CIRCUIT

QUBIT_ORDER, CIRCUIT = load_cirq_circuit(CIRCUIT_FILE)
print(f'Loaded {os.path.basename(CIRCUIT_FILE)}')
print(f'  qubits  = {len(QUBIT_ORDER)}')
print(f'  moments = {len(CIRCUIT)}')


## 3. Translate the Cirq circuit into a tetron MBQC circuit

Strategy:

- Allocate 24 qubits (T1..T24 mapped to Qiskit indices 0..23) and one large
  classical register `c_int` to hold every MBQC intermediate measurement.
- Walk every Cirq operation:
  - `(X**0.5)`               -> `gate_sqrt_X` on (data, ancilla).
  - `(Y**0.5)`               -> `gate_sqrt_Y`.
  - `PhasedXPowGate(0.25, 0.5)` -> `gate_sqrt_W`.
  - `Rz(...)`                -> dropped.
  - `FSimGate(...)`          -> `gate_two_qubit` at `(pi/2, pi/6)`.
- A 12-bit `c_out` register holds the final data-qubit measurements in the
  same order as `QUBIT_ORDER`.


In [ ]:
FSIM_THETA = np.pi / 2
FSIM_PHI   = np.pi / 6


def _is_sqrt_X(g):
    return isinstance(g, cirq.XPowGate) and np.isclose(g.exponent, 0.5)

def _is_sqrt_Y(g):
    return isinstance(g, cirq.YPowGate) and np.isclose(g.exponent, 0.5)

def _is_sqrt_W(g):
    return (isinstance(g, cirq.PhasedXPowGate)
            and np.isclose(g.phase_exponent, 0.25)
            and np.isclose(g.exponent, 0.5))


def count_classical_bits(cirq_circuit,
                         fsim_theta=FSIM_THETA, fsim_phi=FSIM_PHI):
    """Total intermediate-measurement bits the MBQC translation needs."""
    fsim_bits = MBQCTranslatedGates.n_bits_two_qubit(fsim_theta, fsim_phi)
    n = 0
    for moment in cirq_circuit:
        for op in moment.operations:
            g = op.gate
            if   _is_sqrt_X(g): n += MBQCTranslatedGates.N_BITS['sqrt_X']
            elif _is_sqrt_Y(g): n += MBQCTranslatedGates.N_BITS['sqrt_Y']
            elif _is_sqrt_W(g): n += MBQCTranslatedGates.N_BITS['sqrt_W']
            elif isinstance(g, cirq.ZPowGate):  pass         # dropped
            elif isinstance(g, cirq.FSimGate):  n += fsim_bits
    return n


def build_mbqc_tetron_circuit(qubit_order, cirq_circuit,
                              fsim_theta=FSIM_THETA, fsim_phi=FSIM_PHI):
    """24-qubit Qiskit MBQC translation of the Cirq circuit."""
    n_int  = count_classical_bits(cirq_circuit, fsim_theta, fsim_phi)
    n_data = len(qubit_order)

    qr    = QuantumRegister(24,    'q')
    c_int = ClassicalRegister(n_int, 'c_int')
    c_out = ClassicalRegister(n_data, 'c_out')
    qc    = QuantumCircuit(qr, c_int, c_out)

    idx = 0  # rolling offset into c_int

    for moment in cirq_circuit:
        for op in moment.operations:
            g    = op.gate
            grid = [(q.row, q.col) for q in op.qubits]

            if _is_sqrt_X(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                MBQCTranslatedGates.gate_sqrt_X(qc, a, d, c_int, start_idx=idx)
                idx += MBQCTranslatedGates.N_BITS['sqrt_X']

            elif _is_sqrt_Y(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                MBQCTranslatedGates.gate_sqrt_Y(qc, a, d, c_int, start_idx=idx)
                idx += MBQCTranslatedGates.N_BITS['sqrt_Y']

            elif _is_sqrt_W(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                MBQCTranslatedGates.gate_sqrt_W(qc, a, d, c_int, start_idx=idx)
                idx += MBQCTranslatedGates.N_BITS['sqrt_W']

            elif isinstance(g, cirq.ZPowGate):
                continue   # Rz wrappers dropped

            elif isinstance(g, cirq.FSimGate):
                d1, d2, anc = grid_edge_to_qiskit_indices(grid[0], grid[1])
                MBQCTranslatedGates.gate_two_qubit(
                    qc, d1, anc, d2, c_int,
                    theta=fsim_theta, phi=fsim_phi,
                    Z1=0.0, Z2=0.0, Z3=0.0, Z4=0.0,
                    start_idx=idx,
                )
                idx += MBQCTranslatedGates.n_bits_two_qubit(fsim_theta, fsim_phi)

            else:
                raise ValueError(f'Unhandled gate: {g}')

        qc.barrier()

    for i, q in enumerate(qubit_order):
        d = grid_to_qiskit_index(q.row, q.col)
        qc.measure(d, c_out[i])

    return qc


qc_mbqc = build_mbqc_tetron_circuit(QUBIT_ORDER, CIRCUIT)
print(f'MBQC circuit: {qc_mbqc.num_qubits} qubits, '
      f'{qc_mbqc.num_clbits} clbits, depth = {qc_mbqc.depth()}')


## 4. Build the direct gate-based 12-qubit reference circuit

In [ ]:
def build_direct_circuit(qubit_order, cirq_circuit,
                         fsim_theta=FSIM_THETA, fsim_phi=FSIM_PHI):
    """Plain 12-qubit Qiskit circuit using sqrt(X), sqrt(Y), sqrt(W),
    fSim(pi/2, pi/6). Rz wrappers are dropped to match the MBQC translation."""
    n         = len(qubit_order)
    qubit_map = {q: i for i, q in enumerate(qubit_order)}
    qc        = QuantumCircuit(n, n)

    sqW  = UnitaryGate(sqrt_W_matrix(),                   label='√W')
    fsim = UnitaryGate(fSim_matrix(fsim_theta, fsim_phi), label='fSim')

    for moment in cirq_circuit:
        for op in moment.operations:
            g     = op.gate
            q_idx = [qubit_map[q] for q in op.qubits]

            if   _is_sqrt_X(g):                  qc.sx(q_idx[0])
            elif _is_sqrt_Y(g):                  qc.ry(np.pi / 2, q_idx[0])
            elif _is_sqrt_W(g):                  qc.append(sqW,  [q_idx[0]])
            elif isinstance(g, cirq.ZPowGate):   continue                     # Rz dropped
            elif isinstance(g, cirq.FSimGate):   qc.append(fsim, q_idx)
            else: raise ValueError(f'Unhandled gate: {g}')

        qc.barrier()

    for i in range(n):
        qc.measure(i, i)
    return qc


qc_direct = build_direct_circuit(QUBIT_ORDER, CIRCUIT)
print(f'Direct circuit: {qc_direct.num_qubits} qubits, '
      f'{qc_direct.num_clbits} clbits, depth = {qc_direct.depth()}')


## 5. Remove final measurements and prepare statevector simulations

The direct circuit is unitary after removing the final output measurements, so its statevector is deterministic.

The MBQC circuit still contains mid-circuit measurements and feed-forward corrections. Therefore, one statevector run corresponds to one random MBQC measurement trajectory. If the feed-forward rules are correct, the final logical/data state should agree with the direct state for every trajectory, up to numerical precision and global phase.


In [ ]:
# Remove only the final output measurements.
# This preserves the MBQC mid-circuit measurements and feed-forward conditionals.
qc_direct_nom = qc_direct.remove_final_measurements(inplace=False)
qc_mbqc_nom   = qc_mbqc.remove_final_measurements(inplace=False)

print('Direct without final measurements:')
print(f'  qubits = {qc_direct_nom.num_qubits}, clbits = {qc_direct_nom.num_clbits}, depth = {qc_direct_nom.depth()}')
print('MBQC without final data measurements:')
print(f'  qubits = {qc_mbqc_nom.num_qubits}, clbits = {qc_mbqc_nom.num_clbits}, depth = {qc_mbqc_nom.depth()}')

# Approximate memory for one complex128 statevector.
mem_gib = (2 ** qc_mbqc_nom.num_qubits) * 16 / 1024**3
print(f'Approximate MBQC statevector memory: {mem_gib:.2f} GiB')


## 6. Compute the direct-circuit target state

This is the 12-qubit target state in the same order as `QUBIT_ORDER`.


In [ ]:
psi_direct = Statevector.from_instruction(qc_direct_nom)

print('Direct target state:')
print(f'  num_qubits = {int(np.log2(len(psi_direct.data)))}')
print(f'  dimension  = {len(psi_direct.data)}')
print(f'  norm       = {np.linalg.norm(psi_direct.data):.12f}')


## 7. Helper: logical-subsystem fidelity without building a density matrix

The full MBQC statevector lives on 24 tetrons/qubits. We only compare the 12 logical/data qubits to the 12-qubit direct state.

This computes

$$F=\langle \psi_{\rm direct}|\rho_{\rm MBQC,data}|\psi_{\rm direct}\rangle$$

without explicitly constructing the full density matrix.


In [ ]:
# Data qubits in the 24-site tetron register, ordered to match QUBIT_ORDER.
data_qargs = [grid_to_qiskit_index(q.row, q.col) for q in QUBIT_ORDER]
print('Data qubits in 24-site MBQC register:', data_qargs)


def subsystem_fidelity_with_pure_target(full_sv, target_sv, keep_qargs):
    """
    Compute F = <target| rho_keep |target>, where rho_keep is the reduced state
    of full_sv on keep_qargs.

    This avoids building rho_full or rho_keep explicitly. It reshapes the full
    statevector as

        psi[kept_subsystem_basis, environment_basis]

    and evaluates sum_env |<target|psi_env>|^2.

    Qiskit convention:
    - qubit 0 is the least-significant bit in the statevector index.
    - The order of keep_qargs must match the qubit order of target_sv.
    """
    full_data = np.asarray(full_sv.data)
    target    = np.asarray(target_sv.data)

    n_total = int(round(np.log2(full_data.size)))
    n_keep  = len(keep_qargs)

    if full_data.size != 2 ** n_total:
        raise ValueError('full_sv length is not a power of two.')
    if target.size != 2 ** n_keep:
        raise ValueError(
            f'target state has dimension {target.size}, but keep_qargs has {n_keep} qubits.'
        )
    if len(set(keep_qargs)) != len(keep_qargs):
        raise ValueError('keep_qargs contains duplicate qubit indices.')
    if any(q < 0 or q >= n_total for q in keep_qargs):
        raise ValueError('keep_qargs contains an index outside the full statevector.')

    env_qargs = [q for q in range(n_total) if q not in keep_qargs]

    # Axis i corresponds to Qiskit qubit i when using Fortran order.
    tensor = full_data.reshape([2] * n_total, order='F')

    # Put the logical/data axes first, in the same order as the direct target.
    tensor = np.transpose(tensor, keep_qargs + env_qargs)

    # Columns are environment branches. Rows are logical/data basis states.
    psi_mat = tensor.reshape((2 ** n_keep, 2 ** (n_total - n_keep)), order='F')

    amps = target.conj() @ psi_mat
    fidelity = np.sum(np.abs(amps) ** 2)
    return float(np.real_if_close(fidelity))


## 8. Run MBQC statevector trajectories and compare to the direct state

Each trajectory samples the random intermediate MBQC measurement outcomes. A correct feed-forward implementation should give fidelity close to 1 for every trajectory.

Start with a small number of trajectories. Increase `N_TRAJECTORIES` after the first validation succeeds.


In [ ]:
N_TRAJECTORIES = 5
SEED0 = 1234

backend_sv = AerSimulator(method='statevector')

# Save the final post-measurement state after all MBQC feed-forward operations.
qc_mbqc_sv = qc_mbqc_nom.copy()
qc_mbqc_sv.save_statevector('psi_mbqc')

mbqc_t_sv = transpile(qc_mbqc_sv, backend_sv, optimization_level=0)

fidelities = []
infidelities = []

for k in range(N_TRAJECTORIES):
    seed = SEED0 + k
    result = backend_sv.run(mbqc_t_sv, shots=1, seed_simulator=seed).result()
    psi_mbqc_full = Statevector(result.data(0)['psi_mbqc'])

    F = subsystem_fidelity_with_pure_target(
        full_sv=psi_mbqc_full,
        target_sv=psi_direct,
        keep_qargs=data_qargs,
    )
    fidelities.append(F)
    infidelities.append(max(0.0, 1.0 - F))
    print(f'trajectory {k:02d}, seed={seed}: fidelity = {F:.12f}, infidelity = {1.0 - F:.3e}')

print('\nSummary:')
print(f'  min fidelity  = {min(fidelities):.12f}')
print(f'  mean fidelity = {np.mean(fidelities):.12f}')
print(f'  max infidelity = {max(infidelities):.3e}')


In [ ]:
# Plot trajectory-by-trajectory infidelity.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(N_TRAJECTORIES), infidelities, marker='o')
ax.set_xlabel('MBQC measurement trajectory')
ax.set_ylabel('1 - logical fidelity')
ax.set_title('MBQC logical-state agreement with direct circuit')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Notes

- This notebook is intended for **exact debugging**, not for emulating hardware sampling cost.
- The sampler-count comparison is useful later as an experiment-like check, but it contains shot noise and, on a local Aer simulator, may still rely on classical state evolution internally.
- The density matrix is not needed here. The full 24-qubit statevector is large but still far more feasible than a 24-qubit density matrix.
- If some trajectories have low fidelity while others have high fidelity, the likely issue is a feed-forward rule, classical-bit indexing convention, or logical/data-qubit ordering mismatch.
- If all trajectories fail with the same fidelity, check whether the direct circuit and MBQC circuit are truly applying the same effective gates, especially the dropped `Rz(...)` wrappers and the fixed `fSim(pi/2, pi/6)` assumption.
